In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler

# ----------
# Preprocessing Training Set
# ----------

# 1) Load Data Set, Remove Duplicates (9 Duplicates)
train = pd.read_csv('../data/raw/train_data.csv')
train = train.drop(columns = ['id'])
train = train.drop_duplicates()

# 2) Encoding Target
le_target = LabelEncoder()
train['Heart Disease'] = le_target.fit_transform(train['Heart Disease'])

# Storing Statistical Data
median_age = train['Age'].median()
mode_gender = train['Gender'].mode()[0]
mode_work = train['work_type'].mode()[0]

train['work_type'] = train['work_type'].replace('children', 'Never_worked')

# 4) Handle missing values
train['Age'].fillna(median_age, inplace=True)
train['Gender'].fillna(mode_gender, inplace=True)
train['work_type'].fillna(mode_work, inplace=True)
train['smoking_status'].fillna('Unknown', inplace=True)

# 5) Handle outliers using IQR
bounds = {}
columns = ['Age','BP','Cholesterol','Max HR','ST depression']

for column in columns:
    Q1 = train[column].quantile(0.25)
    Q3 = train[column].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    bounds[column] = (lower_bound, upper_bound)

    for counter in train.index:
        if train.loc[counter, column] < lower_bound:
            train.loc[counter, column] = lower_bound
        elif train.loc[counter, column] > upper_bound:
            train.loc[counter, column] = upper_bound

# 6) Encode categorical
categorical_columns = ['Gender', 'work_type','smoking_status']
train = pd.get_dummies(train, columns =categorical_columns)

# 7) Scaling
scaler = StandardScaler()
train[columns] = scaler.fit_transform(train[columns])

# ----------
# Preprocessing Testing Set
# ----------

# 1) Load data set, remove duplicates (1 duplicate)
test = pd.read_csv('../data/raw/test_data.csv')
test = test.drop(columns = ['id'])
test = test.drop_duplicates()

# 2) Encoding Target
test['Heart Disease'] = le_target.transform(test['Heart Disease'])

# 3) Handle missing values
test['work_type'] = test['work_type'].replace('children', 'Never_worked')

test['Age'].fillna(median_age, inplace=True)
test['Gender'].fillna(mode_gender, inplace=True)
test['work_type'].fillna(mode_work, inplace=True)
test['smoking_status'].fillna('Unknown', inplace=True)

# 4) Handle outliers using IQR
for column in columns:
    lower_bound, upper_bound = bounds[column]

    for counter in test.index:
        if test.loc[counter, column] < lower_bound:
            test.loc[counter, column] = lower_bound
        elif test.loc[counter, column] > upper_bound:
            test.loc[counter, column] = upper_bound

# 5) Encode categorical
test = pd.get_dummies(test, columns = categorical_columns)
train, test = train.align(test, join='left', axis=1, fill_value=0)

# 6) Scaling
test[columns] = scaler.transform(test[columns])

# ----------
# Save & Download
# ----------

'''train.to_csv("../data/processed/cleaned_train.csv", index=False)
test.to_csv("../data/processed/cleaned_test.csv", index=False)

files.download('../data/processed/cleaned_train.csv')
files.download('../data/processed/cleaned_test.csv')'''

C:\Users\dell\AppData\Local\Temp\ipykernel_18532\4076408339.py:25: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  train['Age'].fillna(median_age, inplace=True)
C:\Users\dell\AppData\Local\Temp\ipykernel_18532\4076408339.py:26: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment us

'train.to_csv("cleaned_train.csv", index=False)\ntest.to_csv("cleaned_test.csv", index=False)\n\nfrom google.colab import files\nfiles.download(\'cleaned_train.csv\')\nfiles.download(\'cleaned_test.csv\')'